In [56]:
import torch 
import torch.nn as nn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

In [57]:
live = pd.read_csv("../data/samples/trial/live_metrics.csv")
verbose = pd.read_csv("../data/samples/trial/verbose_statements.csv")
initial = pd.read_csv("../data/samples/trial/initial_statement.csv")

In [58]:
live = live.drop(columns=['Unnamed: 0'])

In [59]:
response_time = verbose['total_duration'].tolist()
response_time = response_time[:-1]
reset_iters = live[live['Iteration'] == 1].index.tolist()

gpu_util_avg = []
memory_util_avg = []
clock_util_avg = []

for i in range(1, len(reset_iters)):
    temp_metrics = live.dropna()
    gpu_util_prompt= temp_metrics.iloc[reset_iters[i-1]:reset_iters[i], 3].tolist()
    memeory_util_prompt= temp_metrics.iloc[reset_iters[i-1]:reset_iters[i], 9].tolist()
    clock_util_prompt= temp_metrics.iloc[reset_iters[i-1]:reset_iters[i], -2].tolist()
    gpu_util_avg.append(np.average(gpu_util_prompt))
    memory_util_avg.append(np.average(memeory_util_prompt))
    clock_util_avg.append(np.average(clock_util_prompt))

In [60]:
data = []

for (x, y, z, i) in zip(gpu_util_avg, memory_util_avg, clock_util_avg, response_time[:-1]):
    data.append([torch.Tensor([x, y, z]), torch.Tensor([i])])

In [61]:
class BenchMark(nn.Module):
    def __init__(self, input_features, output_features):
        super().__init__()
        self.l1 = nn.Sequential(
            nn.Linear(input_features, 32),
            nn.ReLU()
        )
        self.l2 = nn.Sequential(
            nn.Linear(32, 16),
            nn.ReLU()
        )
        self.l3 = nn.Linear(16, output_features)

    def forward(self, x):
        x = self.l1(x)
        x = self.l2(x)
        x = self.l3(x)
        return x

In [62]:
model = BenchMark(3, 3)

In [145]:
def minimize_avg_time_loss(predicted, nums, ans):
    total_sum = 0
    for i in range(3):
        total_sum += (predicted[0][i]*nums[i]) 
    loss = nn.MSELoss()
    net_loss = loss(total_sum, ans) 
    return net_loss

optimizer = torch.optim.SGD(model.parameters(), lr=0.001)

In [146]:
train_data = data[:50]
test_data = data[50:]

train_dataloader = DataLoader(train_data, batch_size=10, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=10, shuffle=True)

In [147]:
model.train()
total_loss = []
for batch, (X, y) in enumerate(train_dataloader):
    for X_mini, y_mini in zip(X, y):
        preds = model(X)
        loss = minimize_avg_time_loss(preds, X_mini, y_mini)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss.append(loss)
        print("Current Loss: ")
        print(loss)

Current Loss: 
tensor(25.0922, grad_fn=<MseLossBackward0>)
Current Loss: 
tensor(47.6164, grad_fn=<MseLossBackward0>)
Current Loss: 
tensor(8.5717, grad_fn=<MseLossBackward0>)
Current Loss: 
tensor(0.0350, grad_fn=<MseLossBackward0>)
Current Loss: 
tensor(39.6286, grad_fn=<MseLossBackward0>)
Current Loss: 
tensor(13.4546, grad_fn=<MseLossBackward0>)
Current Loss: 
tensor(7.0750, grad_fn=<MseLossBackward0>)
Current Loss: 
tensor(4.5170, grad_fn=<MseLossBackward0>)
Current Loss: 
tensor(18.1310, grad_fn=<MseLossBackward0>)
Current Loss: 
tensor(21.7211, grad_fn=<MseLossBackward0>)
Current Loss: 
tensor(11.6864, grad_fn=<MseLossBackward0>)
Current Loss: 
tensor(0.0253, grad_fn=<MseLossBackward0>)
Current Loss: 
tensor(5.3781, grad_fn=<MseLossBackward0>)
Current Loss: 
tensor(10.7480, grad_fn=<MseLossBackward0>)
Current Loss: 
tensor(6.4445, grad_fn=<MseLossBackward0>)
Current Loss: 
tensor(4.1179, grad_fn=<MseLossBackward0>)
Current Loss: 
tensor(17.2736, grad_fn=<MseLossBackward0>)
Curre